# IJIM — Resistensi Warga terhadap Aplikasi Pemerintah yang Diwajibkan
**Dataset:** IGAR (Isnan & Pardamean, 2025), Mendeley Data, doi:10.17632/7zryc6k76z.3, lisensi CC BY 4.0

**Cara pakai (Colab gratis):**
1. `Runtime → Change runtime type → T4 GPU` (jika tidak tersedia, CPU tetap jalan tetapi lebih lambat).
2. `Runtime → Run all`. Izinkan akses Google Drive saat diminta.
3. **Jika sesi putus atau waktu habis**, cukup `Run all` lagi. Semua hasil tersimpan di
   `MyDrive/IJIM_IGAR/`, dan notebook melanjutkan dari chunk terakhir yang selesai.
4. Notebook berhenti sendiri dengan aman sebelum batas waktu (`TIME_BUDGET_MIN`).

**Isi:** (1) unduh dan verifikasi data → (2) linimasa harian + garis kebijakan → (3) interrupted time series
dan uji placebo → (4) skor konstruk resistensi berbasis embedding (bisa dilanjutkan) → (5) konstruk × kebijakan →
(6) lembar coding manual untuk validasi (Cohen's kappa) → (7) ringkasan.

In [ ]:
# ====== 0. KONFIGURASI (ubah di sini saja) ======
DRIVE_DIR       = '/content/drive/MyDrive/IJIM_IGAR'   # folder hasil di Google Drive
TIME_BUDGET_MIN = 100      # berhenti aman sebelum batas Colab gratis (~2 jam)
CHUNK           = 20000    # baris per chunk embedding (turunkan jika RAM sempit)
BATCH_GPU, BATCH_CPU = 512, 64
MODEL_NAME      = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
SCOPE_APPS      = ['pertamina', 'satusehat', 'JMO', 'mobileJKN']  # app yang di-embed; None = semua
SAMPLE_PER_APP  = None     # mis. 50000 untuk percobaan cepat; None = semua baris
SIM_THRESHOLD   = 0.50     # ambang label zero-shot (eksploratif)
SIM_MARGIN      = 0.05     # selisih minimal skor tertinggi vs kedua
SAVE_EMB        = True     # simpan embedding fp16 per chunk (~8 MB/20rb baris) untuk klasifier terlatih
SEED            = 42
MAX_PLACEBO     = 300      # jumlah maksimum tanggal placebo per uji (semua tanggal sah bila lebih sedikit)

# Tanggal kebijakan (terverifikasi dari berita; tambahkan sendiri jika perlu)
EVENTS = {
    'pertamina': [('2022-06-27', 'Pengumuman wajib daftar MyPertamina untuk Pertalite/Solar'),
                  ('2022-07-01', 'Pendaftaran & uji coba wajib dimulai (11 daerah)')],
    'satusehat': [('2021-08-10', 'PeduliLindungi wajib untuk masuk mal (diumumkan malam 9 Agu)'),
                  ('2021-08-25', 'Pengumuman PeduliLindungi wajib untuk semua moda transportasi'),
                  ('2021-08-28', 'PeduliLindungi wajib untuk perjalanan transportasi (berlaku)')],
}
# Aplikasi kontrol (tanpa kebijakan wajib baru di jendela yang sama) untuk comparative ITS / DiD
CONTROLS = {'pertamina': 'JMO', 'satusehat': 'mobileJKN'}
DATA_URL = 'https://data.mendeley.com/public-files/datasets/7zryc6k76z/files/2cbb1688-aeee-45f6-8a4e-073704716f31/file_downloaded'
DATA_SHA256 = 'a27a0cf3dae752789fa25e97a5fb39b906f5bc2ea3d93891ff652d0168a3a4a7'

In [ ]:
# ====== 1. SETUP: Drive, waktu, log, simpan atomik ======
import os, sys, time, json, gc, hashlib, subprocess, warnings
warnings.filterwarnings('ignore')
T_START = time.time()
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = DRIVE_DIR
else:                                   # untuk uji lokal
    BASE = os.environ.get('IJIM_BASE', './IJIM_IGAR')
for sub in ['', 'raw', 'chunks', 'fig', 'out']:
    os.makedirs(os.path.join(BASE, sub), exist_ok=True)
P = lambda *a: os.path.join(BASE, *a)

def elapsed_min(): return (time.time() - T_START) / 60
def time_left():   return TIME_BUDGET_MIN - elapsed_min()
def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')} | {elapsed_min():5.1f} mnt] {msg}"
    print(line)
    with open(P('log.txt'), 'a', encoding='utf-8') as f: f.write(line + '\n')
def save_json(obj, path):               # tulis ke .tmp lalu rename: aman jika sesi putus
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f: json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    os.replace(tmp, path)
def save_parquet(df, path):
    tmp = path + '.tmp'; df.to_parquet(tmp, index=False); os.replace(tmp, path)

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers', 'statsmodels', 'pyarrow'], check=False)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg') if not IN_COLAB else None
import matplotlib.pyplot as plt
log(f'Mulai. BASE={BASE}  Colab={IN_COLAB}')

In [ ]:
# ====== 2. UNDUH + VERIFIKASI + KONVERSI KE PARQUET (sekali saja) ======
import requests
RAW_CSV, PARQ = P('raw', 'Rating_labeled.csv'), P('raw', 'igar.parquet')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

if not os.path.exists(PARQ):
    if not os.path.exists(RAW_CSV) or sha256(RAW_CSV) != DATA_SHA256:
        log('Mengunduh IGAR (~96 MB)...')
        with requests.get(DATA_URL, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(RAW_CSV + '.tmp', 'wb') as f:
                for b in r.iter_content(1 << 20): f.write(b)
        os.replace(RAW_CSV + '.tmp', RAW_CSV)
    assert sha256(RAW_CSV) == DATA_SHA256, 'SHA-256 tidak cocok: unduh ulang.'
    log('SHA-256 cocok. Konversi ke parquet (per chunk, hemat RAM)...')
    parts = []
    for ch in pd.read_csv(RAW_CSV, chunksize=100_000, encoding='utf-8',
                          usecols=['Unnamed: 0', 'app', 'content', 'score', 'at', 'appVersion']):
        ch = ch.rename(columns={'Unnamed: 0': 'rid'})
        ch['at'] = pd.to_datetime(ch['at'], errors='coerce')
        ch['score'] = ch['score'].astype('int8')
        ch['content'] = ch['content'].fillna('').astype(str)
        parts.append(ch)
    df = pd.concat(parts, ignore_index=True); del parts
    df['app'] = df['app'].astype('category')
    save_parquet(df, PARQ); del df; gc.collect()
    if IN_COLAB: os.remove(RAW_CSV)    # hemat kuota Drive
log('Data siap: ' + PARQ)

In [ ]:
# ====== 3. PROFIL + LINIMASA HARIAN DENGAN GARIS KEBIJAKAN ======
meta = pd.read_parquet(PARQ, columns=['rid', 'app', 'score', 'at'])
prof = meta.groupby('app', observed=True).agg(n=('score', 'size'), mulai=('at', 'min'), akhir=('at', 'max'),
                                             rating=('score', 'mean'), bintang1=('score', lambda s: (s == 1).mean()))
prof.to_csv(P('out', '01_profil_app.csv')); print(prof.round(3).to_string())

meta['date'] = meta['at'].dt.normalize()
daily = (meta.groupby(['app', 'date'], observed=True)
             .agg(n=('score', 'size'), rating=('score', 'mean'), bintang1=('score', lambda s: (s == 1).mean()))
             .reset_index())
daily['log_n'] = np.log1p(daily.n)
daily.to_csv(P('out', '02_harian.csv'), index=False)

for app, evs in EVENTS.items():
    d = daily[daily.app == app].set_index('date')
    lo = pd.Timestamp(evs[0][0]) - pd.Timedelta(days=90)
    d = d[d.index >= lo]
    fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    ax[0].bar(d.index, d.n, color='#6b7280'); ax[0].set_ylabel('Ulasan/hari'); ax[0].set_yscale('log')
    ax[1].plot(d.index, d.rating, color='#1f4e79'); ax[1].set_ylabel('Rating rata-rata'); ax[1].set_ylim(1, 5)
    for j, (dt, lab) in enumerate(evs, 1):
        for a in ax: a.axvline(pd.Timestamp(dt), color='#b91c1c', ls='--', lw=1, label=f'E{j}: {dt} {lab}' if a is ax[1] else None)
        ax[0].text(pd.Timestamp(dt), ax[0].get_ylim()[1], f' E{j}', fontsize=8, va='top', color='#b91c1c')
    ax[1].legend(loc='lower left', fontsize=7, frameon=False)
    ax[0].set_title(f'{app}: volume dan rating harian di sekitar kebijakan wajib')
    fig.tight_layout(); fig.savefig(P('fig', f'03_linimasa_{app}.png'), dpi=200); plt.close(fig)
log('Profil & linimasa tersimpan di out/ dan fig/')

In [ ]:
# ====== 4. ITS + COMPARATIVE ITS (DiD vs aplikasi kontrol) + UJI PLACEBO ======
import statsmodels.formula.api as smf

def its(d, t0, pre=60, post=60, y='rating', w=True):
    t0 = pd.Timestamp(t0)
    s = d[(d.date >= t0 - pd.Timedelta(days=pre)) & (d.date < t0 + pd.Timedelta(days=post))].copy()
    if s.date.lt(t0).sum() < 14 or s.date.ge(t0).sum() < 3: return None
    s['t'] = (s.date - t0).dt.days; s['post'] = (s.t >= 0).astype(int); s['t_post'] = s.t * s.post
    s['log_n'] = np.log1p(s.n)
    f = f'{y} ~ t + post' + (' + t_post' if s.post.sum() >= 10 else '')   # slope pasca hanya jika data cukup
    m = smf.wls(f, s, weights=s.n) if (w and y != 'log_n') else smf.ols(f, s)   # bobot = jumlah ulasan/hari
    r = m.fit(cov_type='HAC', cov_kwds={'maxlags': 7})
    return dict(level=r.params['post'], se=r.bse['post'], p=r.pvalues['post'], n_pre=int((s.post == 0).sum()), n_post=int(s.post.sum()))

def did_its(df, app_t, app_c, t0, y, **kw):
    # Comparative ITS (DiD harian): outcome = treated - kontrol pada tanggal yang sama
    t = df[df.app == app_t][['date', y, 'n']].rename(columns={y: 'yt', 'n': 'nt'})
    c = df[df.app == app_c][['date', y, 'n']].rename(columns={y: 'yc', 'n': 'nc'})
    m = t.merge(c, on='date')
    if m.empty: return None
    m['diff'] = m.yt - m.yc; m['n'] = np.minimum(m.nt, m.nc)
    return its(m, t0, y='diff', w=False, **kw)

def its_rows(df, ycols, rng=None, placebo=True):
    # ITS satu-aplikasi + DiD terhadap kontrol, dengan placebo di periode pra
    rows = []
    for app, evs in EVENTS.items():
        d = df[df.app == app].sort_values('date'); ctrl = CONTROLS.get(app)
        first = pd.Timestamp(evs[0][0])
        # Placebo: SEMUA tanggal pra-kebijakan yang sah (bukan sampel acak 50), dihitung per desain:
        # untuk DiD hanya tanggal ketika aplikasi kontrol juga punya data (cegah placebo 'kosong').
        def eligible(dates):
            dates = pd.Series(sorted(set(dates)))
            return dates[(dates < first - pd.Timedelta(days=30)) & (dates > dates.min() + pd.Timedelta(days=20))]
        cands = {'ITS': eligible(d.date)}
        if ctrl:
            both = set(d.date) & set(df[df.app == ctrl].date)
            cands['DiD'] = eligible(both) if both else pd.Series([], dtype='datetime64[ns]')
        for k in cands:                                   # batasi maks MAX_PLACEBO dengan langkah teratur (deterministik)
            v = cands[k].values
            cands[k] = v[np.linspace(0, len(v) - 1, MAX_PLACEBO).astype(int)] if len(v) > MAX_PLACEBO else v
        for dt, lab in evs:
            for y in ycols:
                for design in ['ITS'] + (['DiD'] if ctrl else []):
                    f = (lambda t0, **k: its(d, t0, y=y, **k)) if design == 'ITS' else \
                        (lambda t0, **k: did_its(df, app, ctrl, t0, y, **k))
                    r = f(dt)
                    if r is None: continue
                    pick = cands.get(design, []) if placebo else []
                    plc = [rp['level'] for pdt in pick
                           if (rp := f(pdt, post=min(30, (first - pd.Timestamp(pdt)).days)))]
                    p_plc = (np.sum(np.abs(plc) >= abs(r['level'])) + 1) / (len(plc) + 1) if plc else np.nan
                    rows.append(dict(app=app, kontrol=ctrl if design == 'DiD' else '', desain=design, event=dt,
                                     keterangan=lab, outcome=y, **r, n_placebo=len(plc), p_placebo=p_plc))
    return pd.DataFrame(rows)

rng = np.random.default_rng(SEED)
its_tab = its_rows(daily, ['rating', 'bintang1', 'log_n'], rng=rng)
its_tab.to_csv(P('out', '04_its_rating_volume.csv'), index=False)
print(its_tab.round(4).to_string())
log('ITS + DiD + placebo selesai -> out/04_its_rating_volume.csv')

In [ ]:
# ====== 4B. DUPLIKASI & KOORDINASI: apakah lonjakan = suara kolektif/terkoordinasi? ======
import re as _re
txt = pd.read_parquet(PARQ, columns=['rid', 'app', 'content', 'at'])
txt = txt[txt.app.isin(list(EVENTS) + list(CONTROLS.values()))]
txt['date'] = txt['at'].dt.normalize()
norm = txt.content.str.lower().str.replace(r'[^a-z0-9 ]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
txt['nwords'] = norm.str.split().str.len()
txt['norm'] = norm.where(txt.nwords >= 4)                       # teks pendek ("bagus", "ok") wajar kembar, abaikan
first_seen = txt.dropna(subset=['norm']).groupby('norm').date.transform('min')
txt.loc[first_seen.index, 'dup_awal'] = (txt.loc[first_seen.index, 'date'] > first_seen)   # teks >=4 kata yang sudah pernah muncul
txt['dup_awal'] = txt['dup_awal'].fillna(False).astype(float)
dupd = (txt.groupby(['app', 'date'], observed=True)
           .agg(n=('rid', 'size'), dup=('dup_awal', 'mean'), panjang=('nwords', 'mean')).reset_index())
dupd.to_csv(P('out', '04B_duplikasi_harian.csv'), index=False)
dup_tab = its_rows(dupd, ['dup', 'panjang'], rng=np.random.default_rng(SEED))
dup_tab.to_csv(P('out', '04B_its_duplikasi.csv'), index=False)
print(dup_tab[['app', 'desain', 'event', 'outcome', 'level', 'p', 'p_placebo']].round(4).to_string())
del txt, norm, first_seen; gc.collect()
log('Analisis duplikasi -> out/04B_*')

In [ ]:
# ====== 5. INSTRUMEN KONSTRUK RESISTENSI (DRAF — WAJIB DIVALIDASI) ======
# Operasionalisasi awal dari literatur resistensi/coping TI (mis. Markus 1983; Lapointe & Rivard 2005;
# Bhattacherjee dkk. 2018; exit-voice Hirschman). Jangkar di bawah adalah DRAF peneliti:
# revisi bersama literatur, lalu uji dengan coding manual (sel 8) sebelum dipakai di naskah.
CONSTRUCTS = {
  'paksaan':    ['kenapa harus wajib pakai aplikasi', 'rakyat dipaksa install aplikasi ini',
                 'beli bensin saja harus pakai aplikasi', 'kok diwajibkan pakai aplikasi'],
  'beban':      ['ribet dan menyusahkan rakyat', 'bikin repot saja', 'mempersulit masyarakat kecil',
                 'prosesnya berbelit belit'],
  'gagal_teknis': ['aplikasi error tidak bisa dibuka', 'gagal daftar terus', 'loading lama server down',
                 'kode otp tidak masuk'],
  'eksklusi':   ['tidak semua orang punya hp android', 'orang tua tidak bisa pakai aplikasi',
                 'di desa sinyal susah', 'kuota internet mahal'],
  'privasi':    ['data pribadi tidak aman', 'takut data bocor', 'minta data terlalu banyak'],
  'ketidakpercayaan': ['pemerintah membohongi rakyat', 'cuma akal akalan cari untung',
                 'modus memperkaya diri', 'aplikasi ini penipuan'],
  'exit_siasat': ['saya uninstall aplikasi ini', 'mending beli bensin eceran saja',
                 'lebih baik antre manual daripada pakai aplikasi', 'pakai akun orang lain saja'],
  'patuh_menerima': ['sangat membantu', 'mudah digunakan', 'terima kasih aplikasinya bagus',
                 'memudahkan urusan saya'],
}
# Penanda leksikal transparan (pelengkap embedding; mudah diaudit reviewer)
import re
LEX = {
  'lex_paksaan': r'\b(wajib|diwajibkan|dipaksa|paksa|harus pake|harus pakai|keharusan)\b',
  'lex_keluar':  r'\b(hapus|uninstal\w*|uninstall\w*|copot|delete)\b',
  'lex_eksklusi': r'\b(hp jadul|tidak punya hp|gak punya hp|ga punya hp|sinyal|kuota|orang tua|lansia|gaptek)\b',
}
save_json({'constructs': CONSTRUCTS, 'lexicon': LEX, 'model': MODEL_NAME, 'threshold': SIM_THRESHOLD},
          P('out', '05_instrumen_konstruk.json'))
log('Instrumen konstruk disimpan (draf).')

In [ ]:
# ====== 6. SKOR KONSTRUK PER CHUNK — BISA DILANJUTKAN (resume) ======
import torch
from sentence_transformers import SentenceTransformer

text = pd.read_parquet(PARQ, columns=['rid', 'app', 'content'])
if SCOPE_APPS: text = text[text.app.isin(SCOPE_APPS)]
if SAMPLE_PER_APP:
    text = text.sample(frac=1, random_state=SEED)
    text = text[text.groupby('app', observed=True).cumcount() < SAMPLE_PER_APP]
text = text.sort_values('rid').reset_index(drop=True)
n_chunks = int(np.ceil(len(text) / CHUNK))
done = {f for f in os.listdir(P('chunks')) if f.startswith('scores_') and f.endswith('.parquet')}
todo = [i for i in range(n_chunks) if f'scores_{i:04d}.parquet' not in done]
log(f'{len(text):,} ulasan, {n_chunks} chunk; sudah {n_chunks - len(todo)}, sisa {len(todo)}')

if todo:
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = SentenceTransformer(MODEL_NAME, device=dev)
    model.max_seq_length = 64
    if dev == 'cuda': model.half()
    bs = BATCH_GPU if dev == 'cuda' else BATCH_CPU
    names = list(CONSTRUCTS)
    cent = np.stack([model.encode(CONSTRUCTS[k], normalize_embeddings=True).mean(0) for k in names])
    cent /= np.linalg.norm(cent, axis=1, keepdims=True)
    log(f'Model dimuat di {dev}, batch={bs}')
    for i in todo:
        if time_left() < 6:
            log(f'⏸ Waktu hampir habis. Berhenti aman di chunk {i}. Jalankan ulang untuk melanjutkan.'); break
        t = time.time()
        part = text.iloc[i * CHUNK:(i + 1) * CHUNK]
        emb = model.encode(part.content.str.slice(0, 400).tolist(), batch_size=bs,
                           normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)
        sim = (emb.astype(np.float32) @ cent.T).astype(np.float16)
        out = pd.DataFrame(sim, columns=[f'sim_{k}' for k in names]); out.insert(0, 'rid', part.rid.values)
        low = part.content.str.lower().values
        for k, pat in LEX.items(): out[k] = [bool(re.search(pat, s)) for s in low]
        if SAVE_EMB: np.save(P('chunks', f'emb_{i:04d}.npy'), emb.astype(np.float16))   # untuk sel 8B
        save_parquet(out, P('chunks', f'scores_{i:04d}.parquet'))   # ditulis terakhir = penanda chunk selesai
        del emb, sim, out, part; gc.collect()
        if dev == 'cuda': torch.cuda.empty_cache()
        log(f'chunk {i + 1}/{n_chunks} selesai ({time.time() - t:.0f} dtk), sisa waktu {time_left():.0f} mnt')
    del model; gc.collect()
del text; gc.collect()
ALL_DONE = len([f for f in os.listdir(P('chunks')) if f.startswith('scores_')]) >= n_chunks
log('SEMUA CHUNK SELESAI' if ALL_DONE else 'Belum selesai: jalankan ulang notebook (Run all) untuk melanjutkan.')

In [ ]:
# ====== 7. KONSTRUK × KEBIJAKAN: pangsa harian + ITS per konstruk ======
files = sorted(f for f in os.listdir(P('chunks')) if f.startswith('scores_') and f.endswith('.parquet'))
sc = pd.concat([pd.read_parquet(P('chunks', f)) for f in files], ignore_index=True)
simcols = [c for c in sc.columns if c.startswith('sim_')]
S = sc[simcols].astype(np.float32).values
srt = np.sort(S, 1); ok = (srt[:, -1] >= SIM_THRESHOLD) & (srt[:, -1] - srt[:, -2] >= SIM_MARGIN)
sc['konstruk'] = np.where(ok, np.array([c[4:] for c in simcols])[S.argmax(1)], 'lainnya')
log(f'Cakupan label zero-shot: {ok.mean():.1%} (sisanya "lainnya")')
sc = sc.drop(columns=simcols).merge(meta[['rid', 'app', 'score', 'date']], on='rid', how='left')
save_parquet(sc, P('out', '07_label_konstruk.parquet'))

dist = pd.crosstab(sc.app, sc.konstruk, normalize='index').round(3)
dist.to_csv(P('out', '07_distribusi_konstruk_per_app.csv')); print(dist.to_string())

share = (pd.get_dummies(sc.set_index(['app', 'date']).konstruk).groupby(level=[0, 1], observed=True).mean()
           .join(sc.groupby(['app', 'date'], observed=True).size().rename('n'))).reset_index()
lexs = sc.groupby(['app', 'date'], observed=True)[[c for c in sc.columns if c.startswith('lex_')]].mean().reset_index()
share = share.merge(lexs, on=['app', 'date'])
share.to_csv(P('out', '07_pangsa_konstruk_harian.csv'), index=False)

its_k = its_rows(share, [c for c in share.columns if c not in ('app', 'date', 'n')], rng=np.random.default_rng(SEED))
its_k.to_csv(P('out', '07_its_konstruk.csv'), index=False)
print(its_k.sort_values(['app', 'event', 'p']).round(4).to_string())

for app, evs in EVENTS.items():
    d = share[share.app == app].set_index('date')
    d = d[d.index >= pd.Timestamp(evs[0][0]) - pd.Timedelta(days=60)]
    cols = [c for c in CONSTRUCTS if c in d.columns]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    for c in cols: ax.plot(d.index, d[c].rolling(3, min_periods=1).mean(), label=c, lw=1.4)
    for dt, _ in evs: ax.axvline(pd.Timestamp(dt), color='#b91c1c', ls='--', lw=1)
    ax.set_ylabel('Pangsa ulasan (rata-rata 3 hari)'); ax.set_title(f'{app}: konstruk resistensi di sekitar kebijakan wajib')
    ax.legend(ncol=4, fontsize=8); fig.tight_layout(); fig.savefig(P('fig', f'07_konstruk_{app}.png'), dpi=200); plt.close(fig)
log('Analisis konstruk selesai -> out/07_*, fig/07_*')

In [ ]:
# ====== 8. VALIDASI: lembar coding manual + Cohen's kappa ======
# (a) Buat lembar coding berstrata (sekali). Dua pengkode mengisi kolom coder1 & coder2 secara independen
#     dengan salah satu label konstruk (atau 'lainnya'), lalu simpan sebagai out/08_coding_sheet_CODED.csv
SHEET, CODED = P('out', '08_coding_sheet.csv'), P('out', '08_coding_sheet_CODED.csv')
if not os.path.exists(SHEET):
    txt = pd.read_parquet(PARQ, columns=['rid', 'content'])
    smp = sc.sample(frac=1, random_state=SEED)
    smp = smp[smp.groupby(['app', 'konstruk'], observed=True).cumcount() < 25]   # maks 25 per app x konstruk (~600-900 baris)
    smp = smp.merge(txt, on='rid')[['rid', 'app', 'date', 'score', 'content', 'konstruk']]
    smp = smp.sample(frac=1, random_state=SEED).rename(columns={'konstruk': 'label_model'})
    smp['coder1'] = ''; smp['coder2'] = ''
    smp.drop(columns=['label_model']).to_csv(SHEET, index=False)          # pengkode tidak melihat label model
    smp[['rid', 'label_model']].to_csv(P('out', '08_label_model_tersembunyi.csv'), index=False)
    log(f'Lembar coding dibuat: {len(smp)} baris -> {SHEET}'); del txt
if os.path.exists(CODED):
    from sklearn.metrics import cohen_kappa_score, classification_report
    c = pd.read_csv(CODED).merge(pd.read_csv(P('out', '08_label_model_tersembunyi.csv')), on='rid')
    c = c[(c.coder1.astype(str).str.len() > 0) & (c.coder2.astype(str).str.len() > 0)]
    k = cohen_kappa_score(c.coder1, c.coder2); log(f"Cohen's kappa antar-pengkode = {k:.3f} (n={len(c)})")
    agree = c[c.coder1 == c.coder2]
    print(classification_report(agree.coder1, agree.label_model, zero_division=0))
    save_json({'kappa': k, 'n': len(c), 'n_agree': len(agree)}, P('out', '08_validasi.json'))
else:
    print('Belum ada file CODED: isi dulu 08_coding_sheet.csv bersama pengkode kedua.')

In [ ]:
# ====== 8B. KLASIFIER TERLATIH (ukuran konstruk final untuk naskah) ======
# Jalan otomatis setelah 08_coding_sheet_CODED.csv ada. Label = kesepakatan coder1 & coder2.
# Model: regresi logistik di atas embedding (ringan, < 2 menit). Laporan: macro-F1 5-fold + prediksi semua ulasan.
if os.path.exists(CODED) and SAVE_EMB:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.metrics import classification_report, f1_score
    c = pd.read_csv(CODED); c = c[(c.coder1 == c.coder2) & c.coder1.notna()]
    emb_files = sorted(f for f in os.listdir(P('chunks')) if f.startswith('emb_'))
    rid_all = np.concatenate([pd.read_parquet(P('chunks', f.replace('emb_', 'scores_').replace('.npy', '.parquet')),
                                              columns=['rid']).rid.values for f in emb_files])
    pos = pd.Series(np.arange(len(rid_all)), index=rid_all)
    c = c[c.rid.isin(pos.index)]
    X = np.concatenate([np.load(P('chunks', f)) for f in emb_files])[pos[c.rid].values].astype(np.float32)
    y = np.asarray(c.coder1.astype(str).tolist(), dtype=object)   # aman untuk pandas 2 & 3
    vc = pd.Series(y).value_counts(); keep = np.isin(y, vc[vc >= 5].index); X, y = X[keep], y[keep]
    clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=2.0)
    yhat = cross_val_predict(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
    rep = classification_report(y, yhat, zero_division=0); print(rep)
    f1m = f1_score(y, yhat, average='macro'); log(f'Klasifier terlatih: macro-F1 (5-fold) = {f1m:.3f}, n={len(y)}')
    open(P('out', '08B_laporan_klasifier.txt'), 'w').write(rep)
    clf.fit(X, y); del X; gc.collect()
    preds = []
    for f in emb_files:                                    # prediksi per chunk: hemat RAM
        e = np.load(P('chunks', f)).astype(np.float32)
        r = pd.read_parquet(P('chunks', f.replace('emb_', 'scores_').replace('.npy', '.parquet')), columns=['rid'])
        pr = clf.predict_proba(e); r['konstruk_final'] = clf.classes_[pr.argmax(1)]; r['p_max'] = pr.max(1).astype(np.float16)
        preds.append(r); del e
    fin = pd.concat(preds).merge(meta[['rid', 'app', 'score', 'date']], on='rid')
    save_parquet(fin, P('out', '08B_label_final.parquet'))
    share_f = (pd.get_dummies(fin.set_index(['app', 'date']).konstruk_final).groupby(level=[0, 1], observed=True).mean()
                 .join(fin.groupby(['app', 'date'], observed=True).size().rename('n'))).reset_index()
    its_f = its_rows(share_f, [k for k in share_f.columns if k not in ('app', 'date', 'n')], rng=np.random.default_rng(SEED))
    its_f.to_csv(P('out', '08B_its_konstruk_final.csv'), index=False)
    log('ITS konstruk final -> out/08B_its_konstruk_final.csv')
else:
    print('Lewati 8B: tunggu 08_coding_sheet_CODED.csv (coding manual 2 pengkode).')

In [ ]:
# ====== 9. RINGKASAN UNTUK DRAF NASKAH ======
lines = ['# Ringkasan hasil — IJIM IGAR', f'Dibuat: {time.strftime("%Y-%m-%d %H:%M")}', '',
         '## Profil app', prof.round(3).to_markdown(), '', '## ITS & DiD rating/volume (level shift)',
         its_tab[['app', 'desain', 'event', 'outcome', 'level', 'se', 'p', 'p_placebo']].round(4).to_markdown(index=False), '']
if os.path.exists(P('out', '07_its_konstruk.csv')):
    k = pd.read_csv(P('out', '07_its_konstruk.csv'))
    lines += ['## ITS pangsa konstruk (p < .05)', k[k.p < .05][['app', 'desain', 'event', 'outcome', 'level', 'p', 'p_placebo']].round(4).to_markdown(index=False), '']
lines += ['## Catatan keterbatasan', '- Ulasan Play Store = sampel swaseleksi (bukan populasi pengguna).',
          '- Konstruk berbasis embedding adalah DRAF sampai kappa & akurasi terhadap coding manual dilaporkan.',
          '- IGAR berhenti 2 Jul 2022 untuk MyPertamina: jendela pasca-kebijakan sangat pendek.',
          '- DiD memakai satu aplikasi kontrol per kasus; asumsi tren paralel dicek visual & placebo.']
if os.path.exists(P('out', '04B_its_duplikasi.csv')):
    dd = pd.read_csv(P('out', '04B_its_duplikasi.csv'))
    lines += ['', '## Duplikasi/koordinasi (ITS & DiD)', dd[['app', 'desain', 'event', 'outcome', 'level', 'p', 'p_placebo']].round(4).to_markdown(index=False)]
open(P('out', '09_ringkasan.md'), 'w', encoding='utf-8').write('\n'.join(lines))
log(f'Selesai. Total waktu {elapsed_min():.1f} mnt. Semua hasil di {BASE}')

In [ ]:
# ====== 10. UNDUH HASIL (ZIP) KE KOMPUTER ======
# Default: out/ + fig/ + log.txt (kecil, beberapa MB). Ubah IKUT_CHUNKS=True jika ingin ikut embedding (ratusan MB).
import zipfile
IKUT_CHUNKS = False
stamp = time.strftime('%Y%m%d_%H%M')
zip_path = f'/content/IJIM_IGAR_hasil_{stamp}.zip' if IN_COLAB else P(f'IJIM_IGAR_hasil_{stamp}.zip')
folders = ['out', 'fig'] + (['chunks'] if IKUT_CHUNKS else [])
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for sub in folders:
        for root, _, files in os.walk(P(sub)):
            for f in files:
                if f.endswith('.tmp'): continue
                full = os.path.join(root, f)
                z.write(full, os.path.relpath(full, BASE))
    if os.path.exists(P('log.txt')): z.write(P('log.txt'), 'log.txt')
log(f'ZIP siap: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)          # browser akan mengunduh otomatis